In [6]:
pip install pandas numpy scikit-learn implicit tqdm

  Using cached pandas-2.3.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached numpy-2.3.0-cp312-cp312-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached scikit_learn-1.7.0-cp312-cp312-macosx_12_0_arm64.whl.metadata (31 kB)
  Using cached implicit-0.7.2.tar.gz (70 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached scipy-1.15.3-cp312-cp312-macosx_14_0_arm64.whl.metadata (61 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 9.8 MB/s eta 0:00:00a 0:00:01
Using cached numpy-2.3.0-cp312-cp312-macosx_14_0_arm64.whl (5.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 7.0 MB/s eta 0:00:0000:0

In [ ]:
import pandas as pd

train_data = pd.read_csv('hse_train.csv')
# тут пробовал как-то завязаться на количество взаимодействий с айтемами, но отложил эту идею.
# train_data['rank'] = train_data.groupby('user_id')['timestamp'].rank(method='dense', ascending=True)
# train_data['weight'] = 1 / train_data['rank']
# interaction_counts = train_data.groupby(['user_id', 'item_id']).size().reset_index(name='count')
# interaction_counts = interaction_counts.merge(train_data[['user_id', 'item_id', 'weight']], on=['user_id', 'item_id'])
# interaction_counts['final_weight'] = interaction_counts['count'] * interaction_counts['weight']
# train_data = interaction_counts
train_data.head()

,user_id,item_id,timestamp
0,258671,74254,1511701649
1,258671,115615,1511841435
2,258671,176624,1512105022
3,240498,45484,1511605442
4,240498,39504,1511756830


In [ ]:
train_data = train_data.drop_duplicates(subset=['user_id', 'item_id'])

In [92]:
assert train_data.duplicated(subset=['user_id', 'item_id']).sum() == 0

In [93]:
train_data.shape

(4630328, 3)

In [ ]:
from scipy.sparse import coo_matrix
from scipy.sparse import csr_matrix
import numpy as np

user_ids = train_data['user_id'].unique()
item_ids = train_data['item_id'].unique()

user_to_idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
item_to_idx = {item_id: idx for idx, item_id in enumerate(item_ids)}

train_data['user_idx'] = train_data['user_id'].map(user_to_idx)
train_data['item_idx'] = train_data['item_id'].map(item_to_idx)

interaction_matrix = coo_matrix(
    (np.ones(len(train_data)), (train_data['user_idx'], train_data['item_idx'])),
    shape=(len(user_ids), len(item_ids))
)

# interaction_matrix = coo_matrix(
#     (train_data['weight'], (train_data['user_idx'], train_data['item_idx'])),
#     shape=(len(user_ids), len(item_ids))
# )

# interaction_matrix = coo_matrix(
#     (train_data['final_weight'], 
#      (train_data['user_idx'], train_data['item_idx'])),
#     shape=(len(user_ids), len(item_ids))
# )

In [95]:
user_ids.shape

(701981,)

In [96]:
item_ids.shape

(180599,)

In [ ]:
idx_to_item = {v: k for k, v in item_to_idx.items()}

def recommend_for_user(user_idx, model, interaction_matrix, top_n=20):
    user_items = interaction_matrix[user_idx]
    try:
        item_ids, _ = model.recommend(user_idx, user_items, N=top_n)
    except IndexError as e:
        print(f"IndexError while making recommendations for user {user_idx}: {e}")
        return []
    return [idx_to_item[item_idx] for item_idx in item_ids]

In [ ]:
def calculate_ndcg(rec_items, true_items):
    relevance = [1 if item in true_items else 0 for item in rec_items[:20]]
    
    dcg = sum((2**rel - 1) / np.log2(i + 2) for i, rel in enumerate(relevance))
    
    ideal_relevance = sorted(relevance, reverse=True)
    idcg = sum((2**rel - 1) / np.log2(i + 2) for i, rel in enumerate(ideal_relevance))
    
    return dcg / idcg if idcg > 0 else 0

In [ ]:
ts_time = train_data['timestamp'].quantile(0.75)
train_data2 = train_data[train_data['timestamp'] <= ts_time]
test_data = train_data[train_data['timestamp'] > ts_time]

In [ ]:
from implicit.als import AlternatingLeastSquares

train_interaction_matrix = coo_matrix(
    (np.ones(len(train_data2)), (train_data2['user_idx'], train_data2['item_idx'])),
    shape=(len(user_ids), len(item_ids))
).tocsr()

def mean_ndcg_k(model, train_interaction_matrix, test_data, user_ids, user_to_idx):
    ndcg_scores = []
    for user_id in test_data['user_id'].unique():
        user_idx = user_to_idx[user_id]
        true_items = test_data[test_data['user_id'] == user_id]['item_id'].tolist()
        recommended_items = recommend_for_user(user_idx, model, train_interaction_matrix)
        ndcg_scores.append(calculate_ndcg(recommended_items, true_items, k=k))
    return np.mean(ndcg_scores)

best_ndcg = 0
best_params = {}

for factors in [50, 100, 150]:
    for regularization in [0.01, 0.1, 1]:
        model = AlternatingLeastSquares(factors=factors, regularization=regularization, iterations=20, random_state=42)
        model.fit(train_interaction_matrix)
        
        ndcg_score = mean_ndcg_k(model, train_interaction_matrix, test_data, user_ids, user_to_idx)
        print(f"factors={factors}, regularization={regularization}, NDCG@20={ndcg_score}")
        
        if ndcg_score > best_ndcg:
            best_ndcg = ndcg_score
            best_params = {'factors': factors, 'regularization': regularization}

print("Лучшие параметры:", best_params)

100%|██████████| 20/20 [00:44<00:00,  2.25s/it]


factors=50, regularization=0.01, NDCG@20=0.007553705689700812


100%|██████████| 20/20 [00:43<00:00,  2.18s/it]


factors=50, regularization=0.1, NDCG@20=0.007529119117126336


100%|██████████| 20/20 [00:45<00:00,  2.27s/it]


...

factors=150, regularization=0.01, NDCG@20=0.008037758417143015


100%|██████████| 20/20 [03:51<00:00, 11.55s/it]


factors=150, regularization=0.1, NDCG@20=0.008007324693858809


100%|██████████| 20/20 [18:51<00:00, 56.60s/it] 


factors=150, regularization=1, NDCG@20=0.007995215695841456
Лучшие параметры: {'factors': 150, 'regularization': 0.01}


In [ ]:
from implicit.als import AlternatingLeastSquares
# from implicit.bpr import BayesianPersonalizedRanking
# from implicit.lmf import LogisticMatrixFactorization

model = AlternatingLeastSquares(factors=250, regularization=0.01, iterations=20, random_state=42)
# model = BayesianPersonalizedRanking(factors=50, regularization=0.01, iterations=20, random_state=42)

# model = LogisticMatrixFactorization(factors=100, regularization=0.01, iterations=20, random_state=42)
# model.fit(train_interaction_matrix)

model.fit(interaction_matrix.tocsr())

100%|██████████| 20/20 [1:19:51<00:00, 239.60s/it]


In [121]:
recommendations = []
csr_interractions_matrix = interaction_matrix.tocsr()
for user_id in user_ids:
    user_idx = user_to_idx[user_id]
    recs = recommend_for_user(user_idx, model, csr_interractions_matrix)
    for rec in recs:
        recommendations.append({'user_id': user_id, 'items': rec})

submission_df = pd.DataFrame(recommendations)

In [87]:
submission_df.shape

(14039620, 2)

In [122]:
submission_df.to_csv('submission12.csv', index=False)